# M3 — Neuromodulation & stimulation

**NTH bootcamp · Module 3**

Before a cortical visual prosthesis can light up the brain, it has to shape a current waveform that is *safe*, *targetable*, and *audible* to the nervous system. This notebook is the code companion to [`neuromod-and-stim.html`](https://github.com/NeuroTechHub/AIMD_bootcamp/blob/main/modules/neuromod-and-stim.html):

1. Neuromodulation in one minute
2. The five pulse parameters
3. Configure a Utah array
4. Fire the stimulator (mock Ripple)
5. From electrodes to phosphenes — teaser

The HTML page is for moving sliders. This notebook is for the things sliders can't show: the µs↔30 kHz cycle quantisation the hardware actually does, the Shannon-k charge-density math behind every safety chip, the interleaving offsets that let multiple electrodes share a 300 Hz train, and a one-cell bridge into M4's phosphene model.

Exercises are tagged **`[easy]`**, **`[intermediate]`**, or **`[challenge]`**.


## Setup

Required packages:

```bash
pip install --user numpy matplotlib ipywidgets
```

The cell below installs them and defines a tiny **inline mock Ripple** stimulator whose API surface mirrors [`neurolight2.stim.base_stimulator.StimParams`](https://github.com/) and the `create_stimulator("mock_ripple")` driver. No hardware, no `xipppy`, no Trellis — but the call shape and field names are the same, so when you later open the real driver it will look familiar.


In [ ]:
%pip install --user -q numpy matplotlib ipywidgets


In [ ]:
# Imports + inline mock Ripple stimulator + plot_pulse helper.
# Mirror of neurolight2.stim.factory.create_stimulator("mock_ripple").
import math
from dataclasses import dataclass, field
from typing import List, Optional

import numpy as np
import matplotlib.pyplot as plt

CYCLE_US = 33.3           # Ripple Grapevine ticks at 30 kHz (1 cycle = 33.3 us)
SHANNON_K_LIMIT = 1.85    # conservative; microelectrodes routinely exceed
DEFAULT_UTAH_AREA_CM2 = 2.0e-5   # ~1900 um^2 Utah-array tip


@dataclass(frozen=True)
class StimParams:
    '''Per-trial stimulation parameters. All per-electrode lists must have the
    same length. Field names mirror neurolight2.stim.base_stimulator.StimParams.'''
    electrodes: List[int]
    amplitudes_ua: List[int]
    pulse_widths_us: List[float]
    frequencies_hz: List[float]
    num_pulses: List[int]
    interphase_us: float = 60.0
    offsets_us: List[float] = field(default_factory=list)

    def __post_init__(self):
        n = len(self.electrodes)
        for fname, v in [('amplitudes_ua', self.amplitudes_ua),
                         ('pulse_widths_us', self.pulse_widths_us),
                         ('frequencies_hz', self.frequencies_hz),
                         ('num_pulses', self.num_pulses)]:
            if len(v) != n:
                raise ValueError(f'{fname} length {len(v)} != electrodes length {n}')
        if self.offsets_us and len(self.offsets_us) != n:
            raise ValueError(f'offsets_us length {len(self.offsets_us)} != electrodes length {n}')

    def to_dict(self) -> dict:
        return {
            'electrodes': list(self.electrodes),
            'amplitudes_ua': list(self.amplitudes_ua),
            'pulse_widths_us': list(self.pulse_widths_us),
            'frequencies_hz': list(self.frequencies_hz),
            'num_pulses': list(self.num_pulses),
            'interphase_us': self.interphase_us,
            'offsets_us': list(self.offsets_us),
        }


@dataclass
class StimEvent:
    '''What stimulate() returns. Mirrors the safety fields of neurolight2.safety.stim_buffer.StimEvent.'''
    charge_per_phase_nc: float
    charge_density_uc_cm2: float
    shannon_k: float
    duration_ms: float
    safety_ok: bool
    is_executed: bool

    def __str__(self) -> str:
        ok = 'OK' if self.safety_ok else 'BLOCKED'
        return (f'StimEvent[{ok}] Q={self.charge_per_phase_nc:.2f} nC | '
                f'D={self.charge_density_uc_cm2:.0f} uC/cm^2 | '
                f'k={self.shannon_k:.2f} | dur={self.duration_ms:.1f} ms')


class MockRipple:
    '''Hardware-free stimulator with the same call signature as the real driver.

    >>> stim = MockRipple()
    >>> ev = stim.stimulate(params)
    >>> ev.safety_ok
    True
    '''
    def __init__(self, electrode_area_cm2: float = DEFAULT_UTAH_AREA_CM2,
                 shannon_k_limit: float = SHANNON_K_LIMIT):
        self.electrode_area_cm2 = electrode_area_cm2
        self.shannon_k_limit = shannon_k_limit
        self.history: List[StimEvent] = []

    def stimulate(self, params: StimParams) -> StimEvent:
        # Worst-case across electrodes — that's what the safety checker uses.
        worst = max(range(len(params.electrodes)),
                    key=lambda i: params.amplitudes_ua[i] * params.pulse_widths_us[i])
        amp_ua = params.amplitudes_ua[worst]
        pw_us  = params.pulse_widths_us[worst]

        q_nc = amp_ua * pw_us / 1000.0                          # uA * us -> pC * 1000 = nC
        d_uc_cm2 = (q_nc / 1000.0) / self.electrode_area_cm2    # convert nC->uC then per cm^2
        if q_nc > 0 and d_uc_cm2 > 0:
            k = math.log10(q_nc / 1000.0) + math.log10(d_uc_cm2)  # Shannon: log10(Q_uC) + log10(D_uC/cm^2)
        else:
            k = -math.inf

        duration_ms = max(
            (params.num_pulses[i] / params.frequencies_hz[i]) * 1000.0
            for i in range(len(params.electrodes))
        )

        safety_ok = (k <= self.shannon_k_limit)
        ev = StimEvent(
            charge_per_phase_nc=q_nc,
            charge_density_uc_cm2=d_uc_cm2,
            shannon_k=k,
            duration_ms=duration_ms,
            safety_ok=safety_ok,
            is_executed=safety_ok,
        )
        self.history.append(ev)
        return ev


def plot_pulse(amp_ua: float = 80, pw_us: float = 170, interphase_us: float = 60,
               freq_hz: Optional[float] = None, num_pulses: int = 1,
               title: Optional[str] = None, ax=None):
    '''Plot a biphasic train as current vs time (uA vs ms). Cathodic-first.'''
    period_us = 1e6 / freq_hz if freq_hz else 2 * pw_us + interphase_us + 200
    total_us  = period_us * num_pulses
    t = np.linspace(0, total_us, max(2000, int(total_us / 5)))
    i_ua = np.zeros_like(t)
    for k in range(num_pulses):
        t0 = k * period_us
        i_ua[(t >= t0) & (t < t0 + pw_us)] = -amp_ua
        i_ua[(t >= t0 + pw_us + interphase_us) &
             (t <  t0 + pw_us + interphase_us + pw_us)] = +amp_ua
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 2.4))
    ax.plot(t / 1000.0, i_ua, lw=1.3, color='#1c1c1a')
    ax.axhline(0, color='#888', lw=0.5)
    ax.set_xlabel('time (ms)'); ax.set_ylabel('current (uA)')
    ax.set_title(title or f'{num_pulses} biphasic pulse(s) at {amp_ua} uA / {pw_us} us')
    ax.set_xlim(0, total_us / 1000.0)
    ax.grid(True, alpha=0.2)
    if ax is None:
        plt.tight_layout(); plt.show()
    return ax


print('mock-ripple ready · numpy', np.__version__,
      '· Shannon-k limit', SHANNON_K_LIMIT,
      '· electrode area', DEFAULT_UTAH_AREA_CM2, 'cm^2')


## 1 · Neuromodulation in one minute

Electrical stim modulates spike timing by injecting charge across an electrode. Two knobs everyone reaches for first are **amplitude** (how strong) and **frequency** (how often). A 1907 result by Lapicque says the minimum amplitude to fire a neuron falls with pulse width along a strength-duration curve — so amplitude and pulse width together set the *threshold for evoking a spike*.

| Knob | Unit | Typical | What it does |
|---|---|---|---|
| amplitude | µA | 10 – 250 | drives more neurons per pulse |
| pulse width | µs | 50 – 500 | longer phases lower the recruitment threshold |
| frequency | Hz | 10 – 300 | sets the driven firing rate (refractory ceiling ~250 Hz) |
| charge per phase | nC | 5 – 50 | amplitude × pulse width — what safety actually cares about |

The waveform we use is **biphasic, charge-balanced** — a cathodic phase, an interphase gap, then an equal-and-opposite anodic phase. Net DC delivered to the tissue: zero.


In [ ]:
plot_pulse(amp_ua=80, pw_us=170, interphase_us=60,
           title='one biphasic pulse · 80 uA · 170 us · 60 us gap')
plt.tight_layout(); plt.show()


## 2 · The five pulse parameters

The mock — and the real driver — both take a `StimParams` dataclass with five per-electrode lists. The ranges below mirror the HTML page's §02 sliders.

| Parameter | Range | Notes |
|---|---|---|
| `amplitudes_ua` | 10 – 200 µA | per electrode; integer µA |
| `pulse_widths_us` | 50 – 500 µs | per phase, not total pulse |
| `interphase_us` | 0 – 200 µs | gap between the two phases |
| `frequencies_hz` | 10 – 300 Hz | one per electrode in normal mode |
| `num_pulses` | 1 – 100 | how many pulses make the train |

Charge per phase is the headline safety number: `Q_phase = amplitude × pulse_width / 1000` gives nC when the inputs are µA and µs.


In [ ]:
params = StimParams(
    electrodes      = [42],
    amplitudes_ua   = [100],
    pulse_widths_us = [170.0],
    frequencies_hz  = [200.0],
    num_pulses      = [20],
)
print(params.to_dict())


In [ ]:
plot_pulse(amp_ua=100, pw_us=170, interphase_us=60,
           freq_hz=200, num_pulses=4,
           title='train · 100 uA · 170 us · 200 Hz · 4 pulses')
plt.tight_layout(); plt.show()


### Exercise 2.1 — charge per phase `[easy]`

Compute the **charge per phase** for a given amplitude and pulse width, and check it against the **Shannon-k** safety inequality:

```
k = log10(Q_uC) + log10(D_uC_per_cm2)   should stay <= 1.85
```

where `Q_uC` is charge per phase in µC and `D_uC_per_cm2` is the same charge divided by the electrode tip area (Utah-array tip ≈ 1900 µm² → 2.0e-5 cm²). The 1.85 limit is the conservative macroelectrode line — microelectrodes routinely cross it.

1. Write `charge_per_phase_nc(amp_ua, pw_us)` returning nanocoulombs.
2. Write `shannon_k(amp_ua, pw_us, area_cm2=DEFAULT_UTAH_AREA_CM2)` returning k.
3. Print both for `(amp=80, pw=170)` and `(amp=200, pw=300)` — the first should be safely under 1.85, the second over.

> Hint: nC = µA × µs / 1000. µC = nC / 1000.


In [ ]:
# Exercise 2.1 — charge per phase + Shannon k
# You have: DEFAULT_UTAH_AREA_CM2, SHANNON_K_LIMIT, math.log10
#
# def charge_per_phase_nc(amp_ua, pw_us) -> float: ...
# def shannon_k(amp_ua, pw_us, area_cm2=DEFAULT_UTAH_AREA_CM2) -> float: ...

# your code here

# Expected: (80, 170) -> Q ~ 13.6 nC, k ~ 1.05 -> safe
#           (200, 300) -> Q ~ 60.0 nC, k ~ 2.18 -> over limit


### Exercise 2.2 — µs to 30 kHz cycles `[intermediate]`

The Ripple Grapevine schedules everything on a **30,000 Hz tick** — every duration the hardware honours is rounded to the nearest **33.3 µs cycle**. That means you can't ask for a 170 µs pulse and get exactly 170 µs; you get whatever multiple of 33.3 µs is closest. This is the gap between what the slider says and what the electrode actually delivers.

1. Implement `us_to_cycles(us)` that returns the integer cycle count closest to `us` (with a minimum of 1, like the real driver).
2. Implement `cycles_to_us(c)` that goes the other way.
3. Round-trip 170, 60, and 500 µs through both functions; print the original µs, the cycle count, and the quantised µs.

> Hint: `round(us / CYCLE_US)` for the forward direction; multiply for the reverse. The minimum-1-cycle rule prevents zero-length phases.


In [ ]:
# Exercise 2.2 — us to 30 kHz cycles
# Constant available: CYCLE_US = 33.3
#
# def us_to_cycles(us: float) -> int: ...        # round, min 1
# def cycles_to_us(c: int) -> float: ...

# your code here

# Expected (approx):
#   170 us -> 5 cycles -> 166.5 us
#    60 us -> 2 cycles ->  66.6 us
#   500 us -> 15 cycles -> 499.5 us


### Exercise 2.3 — sweep amplitude × pulse width `[intermediate]`, **interactive**

Build a 2D map of charge per phase, with amplitude on the x-axis and pulse width on the y-axis. Overlay the **Shannon-k = 1.85** isoline so you can see at a glance which slider combinations would be blocked.

The `@interact` decorator below gives you sliders for the electrode area (changes the safety frontier — smaller electrodes hit the limit sooner) and the Shannon-k threshold. Fill in the body: build the meshgrid, compute `K` at every point, render `K` with `imshow`, and draw the isoline with `contour`.

> Hint: `A, P = np.meshgrid(amps, pws)` gives you (W, P) grids; `K = np.log10(A*P/1e6) + np.log10((A*P/1e6) / area)`. `ax.contour(A, P, K, levels=[k_limit])` draws the frontier.


In [ ]:
# Exercise 2.3 — interactive amp x pw sweep
import ipywidgets as widgets
from ipywidgets import interact

amps = np.linspace(10, 250, 80)        # uA
pws  = np.linspace(40, 500, 80)        # us

@interact(area_um2=widgets.IntSlider(min=500, max=5000, step=100, value=1900,
                                     description='area (um^2)'),
          k_limit=widgets.FloatSlider(min=1.0, max=2.5, step=0.05, value=1.85,
                                      description='Shannon k'))
def _sweep(area_um2=1900, k_limit=1.85):
    area_cm2 = area_um2 * 1e-8
    # 1) Make the meshgrid of amplitude x pulse width
    # 2) Compute Q in uC and density in uC/cm^2
    # 3) Compute K = log10(Q) + log10(D)
    # 4) imshow(K) with origin='lower', extent=(amps.min, amps.max, pws.min, pws.max)
    # 5) contour(A, P, K, levels=[k_limit]) and label it
    fig, ax = plt.subplots(figsize=(6.5, 4.2))

    # your code here

    ax.set_xlabel('amplitude (uA)'); ax.set_ylabel('pulse width (us)')
    ax.set_title(f'Shannon k  (area={area_um2} um^2, limit={k_limit:.2f})')
    plt.tight_layout(); plt.show()


## 3 · Configure a Utah array

A Utah array is a 10×10 grid of microelectrodes — 96 active sites, 400 µm pitch — implanted in cortex. The mock indexes sites 1–96. The real driver maps each site to a Ripple channel via a fixed lookup (see [`neurolight2.stim.electrode_map`](https://github.com/) for the table). For workshop purposes we treat the site IDs as the addressable handle.

| Attribute | Value |
|---|---|
| sites | 96 active |
| layout | 10 × 10 (corners missing) |
| pitch | 400 µm |
| tip area | ~1900 µm² (= `DEFAULT_UTAH_AREA_CM2`) |


In [ ]:
# Visualise the Utah grid. The four corners of a 10x10 layout are inactive on the
# standard Blackrock/Utah; here we just show the canonical 96 numbered sites.
def utah_grid_positions():
    positions = {}
    site = 1
    for r in range(10):
        for c in range(10):
            if (r, c) in {(0, 0), (0, 9), (9, 0), (9, 9)}:
                continue
            positions[site] = (c, 9 - r)   # x right, y up
            site += 1
    return positions

POS = utah_grid_positions()
xs, ys = zip(*POS.values())
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(xs, ys, s=120, facecolors='#fde7ef', edgecolors='#d86f91')
for sid, (x, y) in POS.items():
    ax.text(x, y, str(sid), ha='center', va='center', fontsize=7, color='#1c1c1a')
ax.set_xticks([]); ax.set_yticks([]); ax.set_xlim(-0.6, 9.6); ax.set_ylim(-0.6, 9.6)
ax.set_aspect('equal'); ax.set_title('Utah array · 96 sites')
plt.tight_layout(); plt.show()


In [ ]:
# Three electrodes, three amplitudes, same train spec.
params3 = StimParams(
    electrodes      = [12, 45, 78],
    amplitudes_ua   = [80, 120, 60],
    pulse_widths_us = [170.0, 170.0, 170.0],
    frequencies_hz  = [200.0, 200.0, 200.0],
    num_pulses      = [10, 10, 10],
)
for i, e in enumerate(params3.electrodes):
    print(f'  site {e:3d}: {params3.amplitudes_ua[i]:3d} uA, '
          f'{params3.pulse_widths_us[i]:.0f} us, {params3.frequencies_hz[i]:.0f} Hz')


### Exercise 3.1 — paint a letter `[easy]`

Given a 10×10 boolean grid representing the letter "C", produce the list of `(site_id, amplitude_ua)` you would need to "draw" it at 100 µA. Use the same `utah_grid_positions()` mapping you saw above to translate grid cells to site IDs.

Sketch of the C pattern (1 = stimulate, 0 = off):

```
0 0 1 1 1 1 1 1 0 0
0 1 0 0 0 0 0 0 1 0
1 0 0 0 0 0 0 0 0 0
1 0 0 0 0 0 0 0 0 0
1 0 0 0 0 0 0 0 0 0
1 0 0 0 0 0 0 0 0 0
1 0 0 0 0 0 0 0 0 0
1 0 0 0 0 0 0 0 0 0
0 1 0 0 0 0 0 0 1 0
0 0 1 1 1 1 1 1 0 0
```

1. Use the provided `letter_C()` to get a `(10, 10)` numpy boolean grid.
2. For each grid cell that is `True`, look up the matching site ID in `POS` (skip the four inactive corners).
3. Build `electrodes` and `amplitudes_ua` lists; wrap them in a `StimParams` with `pw=170`, `freq=200`, `num_pulses=10`.

> Hint: the grid's row 0 is at the top; `POS` puts `y = 9 - row` so y increases upward.


In [ ]:
def letter_C():
    g = np.zeros((10, 10), dtype=bool)
    g[0, 2:8]   = True
    g[9, 2:8]   = True
    g[1, 1]    = g[1, 8] = True
    g[8, 1]    = g[8, 8] = True
    g[2:8, 0]   = True
    return g

C = letter_C()
plt.imshow(C, cmap='Pinks' if 'Pinks' in plt.colormaps() else 'pink_r')
plt.title('letter "C" pattern'); plt.axis('off'); plt.show()


In [ ]:
# Exercise 3.1 — letter to StimParams
# Available: letter_C() -> (10,10) bool; POS: dict[site_id -> (x, y)] with y = 9 - row.
#
# Build a StimParams that stimulates every cell that is True, all at 100 uA,
# 170 us, 200 Hz, 10 pulses.

# your code here  ->  letter_params = StimParams(...)


### Exercise 3.2 — interleaved multi-electrode timing `[intermediate]`

When several electrodes share the same frequency and pulse width, the controller staggers their start times so no two phases collide on the same 30 kHz cycle. The neurolight2 driver computes offsets equally spaced across one period of the train — see [`stim/interleaving.py`](file:///C:/Users/admin/neurolight2/neurolight2/stim/interleaving.py).

Implement `interleave_offsets(electrodes, freq_hz)`:

1. Compute the period: `period_us = 1e6 / freq_hz`.
2. Divide that period into `len(electrodes)` equal slots.
3. Assign offsets in **electrode-ID order** (lowest ID gets slot 0, next gets slot 1, etc), returned in **input order**.
4. The maximum offset must stay below one period.

> Hint: `sorted(range(n), key=lambda i: electrodes[i])` gives the ranks; place each rank at `rank * period / n`.


In [ ]:
# Exercise 3.2 — interleave_offsets
# def interleave_offsets(electrodes: list[int], freq_hz: float) -> list[float]:
#     '''Per-electrode offsets in microseconds, returned in input order.'''
#     ...

# your code here

# Quick check (run after defining):
#   offsets = interleave_offsets([45, 12, 78], freq_hz=300)
#   site 12 is lowest -> rank 0 -> 0 us
#   site 45 -> rank 1 -> 1111 us  (period = 3333 us, /3 slots)
#   site 78 -> rank 2 -> 2222 us
#   returned in input order: [1111, 0, 2222]


And a quick raster to confirm — each row is one electrode, each tick a pulse onset, over a 10 ms window. The staggered start times are what you'd see on a logic analyser tap of the Grapevine.

In [ ]:
# Visualise the interleaved raster (uses your interleave_offsets from 3.2).
# Plot pulse onsets for 6 electrodes at 300 Hz over a 10 ms window.
electrodes = [12, 45, 78, 33, 21, 60]
freq_hz = 300.0
window_ms = 10.0
n_pulses = int(window_ms / 1000 * freq_hz) + 1

# your code here:
#   1) call interleave_offsets(electrodes, freq_hz) -> offsets_us
#   2) for each electrode, compute pulse onsets: offset_us/1000 + k * (1000/freq_hz) for k in 0..n_pulses
#   3) eventplot or vlines over a (n_electrodes,) raster

# fig, ax = plt.subplots(figsize=(8, 3.2))
# ...
# plt.show()


## 4 · Fire the stimulator

Everything above gets us to the call shape the real driver expects: `stim.stimulate(params) -> StimEvent`. The mock doesn't talk to any hardware, but it runs the same safety arithmetic — so a `safety_ok=True` here is a `safety_ok=True` on the real Grapevine too.


In [ ]:
stim = MockRipple()
ev = stim.stimulate(StimParams(
    electrodes      = [42],
    amplitudes_ua   = [100],
    pulse_widths_us = [170.0],
    frequencies_hz  = [200.0],
    num_pulses      = [20],
))
print(ev)
print('history length:', len(stim.history))


### Exercise 4.1 — find the safety frontier `[intermediate]`

Sweep amplitude from 10 → 250 µA at fixed pw = 170 µs and num_pulses = 10. For each amplitude, call `stim.stimulate(...)` on a single-electrode trial and record `event.charge_per_phase_nc` and `event.safety_ok`. Plot charge vs amplitude, colouring points by safety status. Mark the amplitude at which the safety flag flips.

> Hint: instantiate a fresh `MockRipple()` so its history starts empty; `[ev.safety_ok for ev in stim.history]` will give you the boolean trace.


In [ ]:
# Exercise 4.1 — safety frontier sweep
# 1) Make a fresh MockRipple.
# 2) Loop amp in range(10, 251, 10); for each amp call stim.stimulate(...).
# 3) Collect amp, charge_per_phase_nc, safety_ok arrays.
# 4) Scatter charge vs amp, colour by safety_ok.
# 5) Find and print the first amplitude where safety_ok flips False.

# your code here


### Exercise 4.2 — multi-electrode trial `[intermediate]`

Combine §3.1 (letter pattern) and §3.2 (interleaved offsets) into one `StimParams`, fire it through the mock, and confirm both `event.is_executed` and `event.safety_ok` are `True`. Print the returned event.

> Hint: `letter_params` from 3.1 already has the electrodes and amplitudes; just pass the offsets you compute with `interleave_offsets(letter_params.electrodes, freq_hz=200)` to a new `StimParams` constructor.


In [ ]:
# Exercise 4.2 — multi-electrode trial
# Build a StimParams with the letter electrodes + interleaved offsets, fire it.

# your code here


## 5 · From electrodes to phosphenes — teaser

Each stimulated electrode produces one **phosphene** in the visual field. As a first sketch: brightness rises with frequency up to the refractory plateau (~250 Hz) and with amplitude above a threshold of ~30 µA. The full forward model lives in M4; this is just the bridge.

### Exercise 5.1 — drive M4's phosphene model `[challenge]`

1. Implement `phosphene_brightness(amp_ua, freq_hz)` returning a value in [0, 1]:
   - 0 below `amp_thresh = 30` µA
   - linear rise from threshold up to `amp_max = 200` µA
   - multiplied by `min(freq_hz / 250.0, 1.0)` to capture the refractory plateau
2. Render the 10×10 brightness map for your letter pattern.
3. Optional next step: open [`phosphene-simulation.ipynb`](https://github.com/NeuroTechHub/AIMD_bootcamp/blob/main/modules/phosphene-simulation/phosphene-simulation.ipynb) and feed your activations into the real dynaphos forward model.


In [ ]:
# Exercise 5.1 — drive M4's phosphene model
# def phosphene_brightness(amp_ua: float, freq_hz: float) -> float: ...
# Then build a (10, 10) brightness array from letter_params + POS and imshow it.

# your code here


---

**Done.** You've parameterised a biphasic train, quantised it to 30 kHz cycles, drawn a letter on a Utah array, staggered its electrodes to share a single 300 Hz slot, and fired the whole pattern through a Shannon-k safety checker. Swapping the inline mock for the real Grapevine is a one-line import: `from neurolight2.stim.factory import create_stimulator; stim = create_stimulator("mock_ripple")` — same `stimulate(params) -> StimEvent` contract, no other changes.

Module lead: see [`bootcamp-plan.html`](https://github.com/NeuroTechHub/AIMD_bootcamp/blob/main/bootcamp-plan.html). Edit this notebook directly; commit your additions to the bootcamp repo at the end of the day.
